# 06 — Risk Parity / Equal Risk Contribution (ERC)

Notebook 05 found that the Most Diversified Portfolio (MDP) — a purely
decorrelation-driven objective — keeps the UUP corner that survives
covariance shrinkage in notebook 04. This notebook asks the same question
of a third risk-based objective: Equal Risk Contribution (ERC), which
allocates so that every asset contributes the same *share* of total
portfolio risk, $w_i(\Sigma w)_i = 1/N$ for all $i$.

ERC equalizes risk shares *by construction* — that part is an identity,
not a finding, and this notebook says so plainly rather than reporting it
as a result. The actual question is what an equal-risk-share allocation
does to the CASH weight UUP receives, and to concentration (effective N),
relative to GMV and MDP on the same $\Sigma$.

## §1 Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize as opt

import maplab as ml
from maplab import (
    Panel, apply_style, GMV, MaxSharpe, EqualWeight, MostDiversified,
    EqualRiskContribution, risk_contributions, backtest, summary, mdp_closed_form,
)
from maplab.models import _min_variance_long_only, _mdp_long_only, _erc_long_only, _tangency_long_only

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

print(f"prices: {prices.shape[0]:,} days x {prices.shape[1]} assets "
      f"({prices.index.min().date()} -> {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"
print(f"rebalance labels: n={len(rdates)}  first={rdates[0].date()}  last={rdates[-1].date()}")

In [ ]:
# Copied from notebook 04/05 (notebook-local helper, not part of the shared harness).
def ols_with_intercept(y: pd.Series, X: pd.DataFrame) -> dict:
    # OLS y ~ 1 + X via lstsq. Local helper (statsmodels is not installed
    # in this environment, so no Newey-West/HAC correction here). Returns
    # coef/tstat Series indexed ["alpha", *X.columns], R^2, and n.
    y_np = y.to_numpy()
    X_np = X.to_numpy()
    n = len(y_np)
    Xd = np.column_stack([np.ones(n), X_np])
    coefs, _, _, _ = np.linalg.lstsq(Xd, y_np, rcond=None)
    fitted = Xd @ coefs
    resid = y_np - fitted
    dof = n - Xd.shape[1]
    sigma2 = float(resid @ resid) / dof
    XtX_inv = np.linalg.inv(Xd.T @ Xd)
    se = np.sqrt(np.diag(XtX_inv) * sigma2)
    tstats = coefs / se
    ss_tot = float(np.sum((y_np - y_np.mean()) ** 2))
    ss_res = float(resid @ resid)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    names = ["alpha"] + list(X.columns)
    return {
        "coef": pd.Series(coefs, index=names),
        "tstat": pd.Series(tstats, index=names),
        "r2": r2,
        "n": n,
    }


# Copied from notebook 04/05.
def sharpe_se(sr: float, T_years: float) -> float:
    return float(np.sqrt((1 + sr ** 2 / 2) / T_years))


# Notebook-local: half-L1 distance between two weight vectors/frames.
def half_l1(w1, w2) -> float:
    return float(0.5 * (w1 - w2).abs().sum())


# Notebook-local: effective N = 1 / sum(w_i^2).
def effN(w) -> float:
    return float(1.0 / (w ** 2).sum())

In [ ]:
class InverseVol(ml.Strategy):
    # Notebook-local: inverse-vol weights, sample covariance window.
    name = "IV"
    family = "Risk-based"

    def predict_weights(self, panel: Panel, asof: pd.Timestamp) -> pd.Series:
        _, Sigma = self._estimate(panel, asof)
        sigma = np.sqrt(np.diag(Sigma.to_numpy()))
        w = (1.0 / sigma) / np.sum(1.0 / sigma)
        return pd.Series(w, index=Sigma.columns)

## §2 PRE-REGISTRATION

**Registered:** ERC(sample). **Reproduction gate:** GMV(sample), EW,
MDP(sample). **Sensitivity:** ERC(ledoit_wolf). **Reference:** IV (inverse
vol, notebook-local, sample vols on the same 252d window) — never
backtested with a shrinkage estimator, same convention as notebook 05.

**Windows:** full / train (&#8804; 2022-12-31) / test (2023+). Paired test =
daily return difference, t = mean/sd&#183;&#8730;n, plain OLS, no HAC; counts
only at &#124;t&#124; &#8805; 2.

Directions for H1/H2 were fixed by a **read-only snapshot check** at
2014-12-31 / 2017-12-31 / 2022-12-31, run **before** this notebook, on the
sample-covariance ERC/GMV/MDP solutions at those three dates only.

**H1 (corner, direction from the snapshot: SHRINKS):** w_UUP(ERC) &lt;
w_UUP(GMV) AND w_UUP(ERC) &lt; w_UUP(MDP). Supported only if both hold at
2022-12-31 AND on &gt; 50% of the 208 rebalances AND on &gt; 50% of the 40
test rebalances. &#946;<sub>UUP,p</sub> is reported alongside as context, not
as a pass condition.

**H2 (effN, direction from the snapshot: &#8805;):** effN(ERC) &#8805;
max(effN GMV, effN MDP) on &gt; 50% of the 208 rebalances AND &gt; 50% of the
40 test rebalances.

**H3 (null):** ERC &#8722; GMV and ERC &#8722; MDP, **&Delta;&alpha; from the
direct diff-regression**, &#124;t&#124; &lt; 2. The *total* paired difference is
reported with the &#946; decomposition but carries **no directional claim**
— ERC runs at a higher ex-ante vol than GMV on the same &Sigma;, so a raw
return gap between them is expected and is not itself evidence of anything.
Registering H3 on &Delta;&alpha; is what makes the test about the ERC construction
rather than about the vol difference.

**H4 (ERC &#8722; EW &lt; 0, beta-driven):** supported only if the
full-window paired t &#8804; &#8722;2 AND the test-window mean is also
negative AND &#124;&Delta;&#946; part&#124; &gt; &#124;&Delta;&#945;&#124;.
Same rule as notebook 05's H4.

**H5 (nulls):** ERC(LW) &#8722; ERC(S) null (&#124;t&#124; &lt; 2, full and
test); ERC(S) &#8722; IV &Delta;&#945; null (&#124;t&#124; &lt; 2, direct
diff-regression).

A pass on the full window only is reported as "not robust", never
"supported".

**Honest limitation:** the snapshot dates that fixed the H1/H2 directions
are in-sample — 2022-12-31 is inside the train window, and it is also one
of the dates this notebook backtests over. The &gt; 50%-of-**test**
requirement (2023+, entirely out of that snapshot) is the guard against
that circularity; it does not remove the circularity from the 2022-12-31
single-date check itself, which is reported as description, not as a
pass/fail condition on its own.

Note on this cell's H3 wording: the registered statistic (&Delta;&alpha;) is as
fixed in the design before this notebook was built; an earlier draft of this
cell described H3 as a test on the total paired difference, which was a
transcription error, not a choice made after seeing results. Both
statistics are reported in &#167;6 either way, so the reader can apply
whichever reading they prefer.

## §3 Mechanism

- **ERC condition.** $w_i(\Sigma w)_i$ equal for all $i$, $\mathbf 1'w=1$,
  $w>0$. Solved via the strictly convex surrogate
  $\min_{y>0} \tfrac12 y'\Sigma y - \tfrac1N\sum_i\log y_i$, $w=y/\mathbf1'y$,
  whose stationarity condition $y_i(\Sigma y)_i = 1/N$ gives equal risk
  contributions exactly. The log barrier makes $w>0$ automatic — unlike the
  tangency/MDP QPs there is no boundary case and no fallback. The solution
  is unique.
- **Euler's theorem.** $\sum_i RC_i = \sigma_p$ exactly, where
  $RC_i = w_i(\Sigma w)_i/\sigma_p$ — risk is homogeneous of degree 1 in
  $w$, so it decomposes additively across assets with no residual.
- **Identity A** (this is the one that explains the cash weight):
  $w_i\cdot\beta_{i,p} = 1/N$ where $\beta_{i,p} = (\Sigma w)_i / w'\Sigma w$
  is asset $i$'s beta to the portfolio itself. An asset that *hedges* the
  portfolio has small (or negative) $\beta_{i,p}$, so buying its mandated
  $1/N$ share of total risk requires a correspondingly LARGE cash weight.
- **Identity B.** Risk weights $x = w\circ\sigma/(w'\sigma)$ equal ERC run
  on the correlation matrix $C=D^{-1}\Sigma D^{-1}$ — the same shape as
  notebook 05's MDP-on-$C$ result (there it was long-only GMV on $C$). Both
  are scale-invariant: $w(K\Sigma K)$ is proportional to $K^{-1}w(\Sigma)$
  for any diagonal $K>0$.
- **Special cases.** Constant correlation &#8658; inverse vol (any $N$);
  $N=2$ &#8658; inverse vol for any $\rho$.
- **Contrast with long-only GMV.** Every asset GMV actually *holds*
  (i.e. $w_i>0$) has $\beta_{i,p}=1$ exactly — that is GMV's own
  first-order condition restricted to its support. So GMV equalizes
  MARGINAL risk across the assets it holds, while ERC equalizes TOTAL risk
  contribution across ALL assets. This is verified numerically just below,
  not just asserted.

In [ ]:
DATES = [pd.Timestamp(d) for d in ["2014-12-31", "2017-12-31", "2022-12-31"]]

def port_vol(w, Sigma_np):
    w = np.asarray(w)
    return float(np.sqrt(w @ Sigma_np @ w))

def dr(w, sigma_np, Sigma_np):
    w = np.asarray(w)
    return float(w @ sigma_np) / port_vol(w, Sigma_np)

snap_rows = []
gmv_beta_dev = {}
uup_context_rows = []
for asof in DATES:
    mu, Sigma = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof)
    Sigma_np = Sigma.to_numpy()
    sigma_np = np.sqrt(np.diag(Sigma_np))
    iU = list(Sigma.columns).index("UUP")
    n = Sigma_np.shape[0]

    w_gmv = _min_variance_long_only(Sigma)
    w_mdp, _ = _mdp_long_only(Sigma, "snap", asof)
    w_erc, _ = _erc_long_only(Sigma, "snap", asof)

    rf_ann = MaxSharpe()._rf_ann(panel, asof)
    excess = mu - rf_ann
    if excess.max() > 0:
        w_msr, _ = _tangency_long_only(excess.to_numpy(), Sigma_np, "snap", asof)
    else:
        w_msr = _min_variance_long_only(Sigma)

    w_ew = np.full(n, 1.0 / n)
    w_iv = (1.0 / sigma_np) / np.sum(1.0 / sigma_np)

    def row(name, w):
        w = np.asarray(w)
        var = float(w @ Sigma_np @ w)
        beta_p = (Sigma_np @ w) / var
        rc = (w * (Sigma_np @ w)) / np.sqrt(var)
        return {
            "asof": asof.date(), "strategy": name,
            "exante_vol": port_vol(w, Sigma_np), "effN": effN(w),
            "max_w": float(w.max()), "max_ticker": Sigma.columns[int(np.argmax(w))],
            "w_UUP": float(w[iU]), "UUP_rc_share": float(rc[iU] / rc.sum()),
            "UUP_beta_p": float(beta_p[iU]),
            "DR": dr(w, sigma_np, Sigma_np),
        }

    for name, w in [("GMV", w_gmv), ("MDP", w_mdp), ("ERC", w_erc),
                     ("MaxSharpe", w_msr), ("EW", w_ew), ("IV", w_iv)]:
        snap_rows.append(row(name, w))

    beta_gmv = (Sigma_np @ w_gmv) / float(w_gmv @ Sigma_np @ w_gmv)
    held = w_gmv > 1e-8
    gmv_beta_dev[asof.date()] = float(np.max(np.abs(beta_gmv[held] - 1.0)))

    D_inv = np.diag(1.0 / sigma_np)
    C = D_inv @ Sigma_np @ D_inv
    uup_mean_corr = float((C[iU].sum() - 1.0) / (n - 1))
    uup_vol_rank = int((sigma_np <= sigma_np[iU]).sum())
    rho_bar = float((C.sum() - n) / (n * (n - 1)))
    uup_context_rows.append({
        "asof": asof.date(), "uup_mean_pairwise_corr": uup_mean_corr,
        "uup_vol_rank": uup_vol_rank, "rho_bar": rho_bar,
    })

snap_table = pd.DataFrame(snap_rows).set_index(["asof", "strategy"])
print(snap_table.round(4))

uup_context_table = pd.DataFrame(uup_context_rows).set_index("asof")
print("\nUUP context (sample cov): mean pairwise correlation, vol rank (1 = lowest), "
      "mean off-diagonal correlation")
print(uup_context_table.round(4))

print("\nGMV first-order check: max|beta_i,p - 1| over assets with w_i > 1e-8 "
      "(should be ~0; the ~1e-5 floor is SLSQP's weight precision, not a failed identity)")
for d, v in gmv_beta_dev.items():
    print(f"  {d}: {v:.3e}")

### 3.3 &#8212; Figure F1: snapshot map at 2022-12-31

Where do GMV, MDP, ERC, MaxSharpe and EW sit in (vol, &mu;) space at the
train-window snapshot used to fix H1/H2's directions? &mu; here is a
252-day trailing sample mean, so the y-axis is illustrative &#8212; the
honest reading is the x-position (risk) and the printed weights, not
"expected return". MaxSharpe is computed at this single date only (a
snapshot reference, not backtested &#8212; the backtested runs are in
&#167;4).

In [ ]:
asof_2022 = pd.Timestamp("2022-12-31")
mu_2022, Sigma_2022 = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
Sigma_np = Sigma_2022.to_numpy()
mu_np = mu_2022.to_numpy()
n = Sigma_np.shape[0]
tickers = list(Sigma_2022.columns)

w_gmv_2022 = _min_variance_long_only(Sigma_2022)
w_mdp_2022, _ = _mdp_long_only(Sigma_2022, "F1", asof_2022)
w_erc_2022, _ = _erc_long_only(Sigma_2022, "F1", asof_2022)
rf_ann_2022 = MaxSharpe()._rf_ann(panel, asof_2022)
excess_2022 = mu_2022 - rf_ann_2022
w_msr_2022, _ = _tangency_long_only(excess_2022.to_numpy(), Sigma_np, "F1", asof_2022)
w_ew_2022 = np.full(n, 1.0 / n)

def min_variance_at_target(Sigma_np, mu_np, target, x0):
    n = Sigma_np.shape[0]
    ones = np.ones(n)
    cons = [
        {"type": "eq", "fun": lambda w: w.sum() - 1.0, "jac": lambda w: ones},
        {"type": "eq", "fun": lambda w: w @ mu_np - target, "jac": lambda w: mu_np},
    ]
    bounds = [(0.0, 1.0)] * n
    res = opt.minimize(
        lambda w: w @ Sigma_np @ w, x0, jac=lambda w: 2 * Sigma_np @ w, method="SLSQP",
        bounds=bounds, constraints=cons, options={"ftol": 1e-12, "maxiter": 1000},
    )
    return res.x if res.success else None

rng_mc = np.random.default_rng(7)
mc_weights = rng_mc.dirichlet(np.full(n, 0.3), size=20_000)
mc_ret = mc_weights @ mu_np
mc_vol = np.sqrt(np.einsum("ij,jk,ik->i", mc_weights, Sigma_np, mc_weights))

targets = np.linspace(mu_np.min(), mu_np.max(), 60)
x0 = np.full(n, 1.0 / n)
frontier_ret, frontier_vol = [], []
for t in targets:
    w = min_variance_at_target(Sigma_np, mu_np, t, x0)
    if w is None:
        continue
    x0 = w
    frontier_ret.append(w @ mu_np)
    frontier_vol.append(np.sqrt(w @ Sigma_np @ w))
frontier_ret = np.array(frontier_ret)
frontier_vol = np.array(frontier_vol)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(mc_vol, mc_ret, s=2, alpha=0.15, color="#bbbbbb", zorder=0,
           label="random long-only portfolios (Dirichlet a=0.3)")

asset_vol = np.sqrt(np.diag(Sigma_np))
group_order = list(ml.ASSET_GROUPS.keys())
group_colors = dict(zip(group_order, plt.cm.tab10(np.linspace(0, 1, len(group_order)))))
for i, tkr in enumerate(tickers):
    grp = ml.GROUP_OF[tkr]
    ax.scatter(asset_vol[i], mu_np[i], s=30, color=group_colors[grp], zorder=2)
    ax.annotate(tkr, (asset_vol[i], mu_np[i]), fontsize=6,
                xytext=(3, 3), textcoords="offset points",
                color="#c0392b" if tkr == "UUP" else "#666666",
                fontweight="bold" if tkr == "UUP" else "normal")

gmv_ret = float(w_gmv_2022 @ mu_np); gmv_vol = port_vol(w_gmv_2022, Sigma_np)
mdp_ret = float(w_mdp_2022 @ mu_np); mdp_vol = port_vol(w_mdp_2022, Sigma_np)
erc_ret = float(w_erc_2022 @ mu_np); erc_vol = port_vol(w_erc_2022, Sigma_np)
msr_ret = float(w_msr_2022 @ mu_np); msr_vol = port_vol(w_msr_2022, Sigma_np)
ew_ret = float(w_ew_2022 @ mu_np); ew_vol = port_vol(w_ew_2022, Sigma_np)

efficient = frontier_ret >= gmv_ret
ax.plot(frontier_vol[~efficient], frontier_ret[~efficient], color="#999999", ls="--", lw=1.2, zorder=3)
ax.plot(frontier_vol[efficient], frontier_ret[efficient], color="#999999", lw=1.8, zorder=3, label="long-only frontier")

ax.scatter([gmv_vol], [gmv_ret], marker="*", s=200, color=ml.FAMILY_COLORS["Risk-based"], zorder=4, label="GMV")
ax.scatter([mdp_vol], [mdp_ret], marker="P", s=140, color=ml.FAMILY_COLORS["Risk-based"], zorder=4, label="MDP")
ax.scatter([erc_vol], [erc_ret], marker="X", s=160, color="#d4a017", zorder=4, label="ERC")
ax.scatter([msr_vol], [msr_ret], marker="*", s=200, color=ml.FAMILY_COLORS["Return-based"], zorder=4, label="MaxSharpe")
ax.scatter([ew_vol], [ew_ret], marker="D", s=90, color=ml.FAMILY_COLORS["Benchmark"], zorder=4, label="EqualWeight")

ax.set_xlabel("annualized volatility")
ax.set_ylabel("annualized arithmetic expected return (mu)")
ax.set_title(f"Snapshot {asof_2022.date()} -- GMV / MDP / ERC / MaxSharpe / EW")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

iU2022 = tickers.index("UUP")
print(f"w_UUP: GMV={w_gmv_2022[iU2022]:.4f}  MDP={w_mdp_2022[iU2022]:.4f}  ERC={w_erc_2022[iU2022]:.4f}  "
      f"MaxSharpe={w_msr_2022[iU2022]:.4f}  EW={w_ew_2022[iU2022]:.4f}")

### 3.4 &#8212; Figure F2: ERC mechanism grid

Cash weight vs. risk-contribution share, asset by asset, for GMV / MDP /
ERC / EW at the three snapshot dates &#8212; the picture behind Identity A.
Every ERC panel's risk-share bars sit exactly on the dashed 1/13 line by
construction (that is the *definition* of ERC, not a result); the
interesting bars are GMV's and MDP's, and the interesting gap in every
panel is between the risk-share bar and the cash-weight bar for UUP
specifically. The bottom panel checks Identity A directly: at 2022-12-31,
ERC's $w_i$ plotted against $1/(N\beta_{i,p})$ should fall exactly on the
diagonal.

In [ ]:
methods = ["GMV", "MDP", "ERC", "EW"]
tickers = ml.UNIVERSE
iU = tickers.index("UUP")
one_over_n = 1.0 / len(tickers)

grid_rows = []
fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharey=True)
for r, asof in enumerate(DATES):
    _, Sigma = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof)
    Sigma = Sigma.reindex(index=tickers, columns=tickers)
    Sigma_np = Sigma.to_numpy()
    n = Sigma_np.shape[0]

    w_gmv = _min_variance_long_only(Sigma)
    w_mdp, _ = _mdp_long_only(Sigma, "F2", asof)
    w_erc, _ = _erc_long_only(Sigma, "F2", asof)
    w_ew = np.full(n, 1.0 / n)
    weights = {"GMV": w_gmv, "MDP": w_mdp, "ERC": w_erc, "EW": w_ew}

    for c, name in enumerate(methods):
        w = weights[name]
        var = float(w @ Sigma_np @ w)
        rc = (w * (Sigma_np @ w)) / np.sqrt(var)
        rc_share = rc / rc.sum()
        beta_p = (Sigma_np @ w) / var

        ax = axes[r, c]
        x = np.arange(n)
        colors_cash = ["#c0392b" if t == "UUP" else "#1f4e79" for t in tickers]
        colors_rc = ["#c0392b" if t == "UUP" else "#2e7d32" for t in tickers]
        ax.bar(x - 0.2, w, width=0.4, color=colors_cash, alpha=0.9,
               label="cash weight" if r == 0 and c == 0 else None)
        ax.bar(x + 0.2, rc_share, width=0.4, color=colors_rc, alpha=0.6,
               label="risk share" if r == 0 and c == 0 else None)
        ax.axhline(one_over_n, color="black", ls="--", lw=0.8)
        ax.set_title(f"{asof.date()} - {name}", fontsize=8)
        if r == 2:
            ax.set_xticks(x); ax.set_xticklabels(tickers, rotation=90, fontsize=6)
        else:
            ax.set_xticks([])

        grid_rows.append({
            "asof": asof.date(), "method": name,
            "w_UUP": float(w[iU]), "UUP_rc_share": float(rc_share[iU]),
            "UUP_beta_p": float(beta_p[iU]), "effN": effN(w),
        })

fig.suptitle("Cash weight vs. risk-contribution share (dashed = 1/13)")
plt.tight_layout()
plt.show()

grid_table = pd.DataFrame(grid_rows).set_index(["asof", "method"])
print(grid_table.round(4))

In [ ]:
_, Sigma_2022b = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
Sigma_2022b = Sigma_2022b.reindex(index=tickers, columns=tickers)
Sigma_np_2022b = Sigma_2022b.to_numpy()
w_erc_2022b, _ = _erc_long_only(Sigma_2022b, "F2diag", asof_2022)
var_2022b = float(w_erc_2022b @ Sigma_np_2022b @ w_erc_2022b)
beta_2022b = (Sigma_np_2022b @ w_erc_2022b) / var_2022b
rhs = 1.0 / (len(tickers) * beta_2022b)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(rhs, w_erc_2022b, s=30, color=ml.FAMILY_COLORS["Risk-based"])
lims = [min(rhs.min(), w_erc_2022b.min()), max(rhs.max(), w_erc_2022b.max())]
ax.plot(lims, lims, color="black", ls="--", lw=0.8)
ax.set_xlabel("1 / (N * beta_i,p)")
ax.set_ylabel("w_i (ERC)")
ax.set_title(f"Identity A check, {asof_2022.date()}")
plt.tight_layout(); plt.show()

print(f"max|w_i - 1/(N*beta_i,p)| at {asof_2022.date()}: {np.max(np.abs(w_erc_2022b - rhs)):.3e}")

## &#167;4 Backtests

Six fresh instances, same cost/rebalance/warm-up harness as notebooks
01&#8211;05. `ERC(S)` is the registered run; `GMV(S)`, `EW`, `MDP(S)` are
the reproduction-gate comparators (already backtested in earlier
notebooks); `ERC(LW)` is the covariance-estimator sensitivity; `IV` is the
notebook-local inverse-vol reference (never backtested with a shrinkage
estimator, same convention as notebook 05).

In [ ]:
strategies = {
    "ERC(S)": EqualRiskContribution(cov_estimator=ml.sample_cov),
    "GMV(S)": GMV(cov_estimator=ml.sample_cov),
    "EW": EqualWeight(),
    "MDP(S)": MostDiversified(cov_estimator=ml.sample_cov),
    "ERC(LW)": EqualRiskContribution(cov_estimator=ml.ledoit_wolf_cov),
    "IV": InverseVol(),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

print(f"{len(results)} runs completed.")

### Reproduction gate

GMV(S), EW, and MDP(S) must match notebooks 01/05 in full/train/test
Sharpe to 2 dp: GMV 0.62/0.59/0.76, EW 0.74/0.70/0.91, MDP(S)
0.76/0.70/0.98. If this doesn't hold, the harness or estimator wiring
changed something and everything downstream is suspect &#8212; so this
stops the notebook (`assert`) rather than just printing a warning.

In [ ]:
rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel]
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]
summary_table["cost_drag"] = summary_table["ann_turnover"] * ml.COST_BPS / 1e4
summary_table.round(4)

In [ ]:
expected_full_sharpe = {"GMV(S)": 0.62, "EW": 0.74, "MDP(S)": 0.76}
expected_train_sharpe = {"GMV(S)": 0.59, "EW": 0.70, "MDP(S)": 0.70}
expected_test_sharpe = {"GMV(S)": 0.76, "EW": 0.91, "MDP(S)": 0.98}

for name, exp in expected_full_sharpe.items():
    got = round(float(summary_table.loc[(name, "full"), "sharpe"]), 2)
    assert got == exp, f"REPRODUCTION GATE FAILED: {name} full Sharpe {got} != {exp}"
for name, exp in expected_train_sharpe.items():
    got = round(float(summary_table.loc[(name, "train (<= split)"), "sharpe"]), 2)
    assert got == exp, f"REPRODUCTION GATE FAILED: {name} train Sharpe {got} != {exp}"
for name, exp in expected_test_sharpe.items():
    got = round(float(summary_table.loc[(name, "test (> split)"), "sharpe"]), 2)
    assert got == exp, f"REPRODUCTION GATE FAILED: {name} test Sharpe {got} != {exp}"

print("REPRODUCTION GATE: PASSED -- GMV(S), EW, MDP(S) match notebooks 01/05 to 2 dp.")

## &#167;5 Mechanism checks, all 208 rebalances &#215; {sample, Ledoit-Wolf}

Recomputed fresh (not read off the &#167;4 backtests), so this is an
independent check of the solver itself, not a tautology. Tolerance 1e-11
unless noted &#8212; coordinate descent (the solver actually shipped in
`_erc_long_only`) delivers max deviations around 1e-12 per the read-only
design check that ran before this notebook was built, which is why the
tolerance is set at 1e-11.

In [ ]:
mech5_rows = []
for est_name, est_fn in [("sample", ml.sample_cov), ("lw", ml.ledoit_wolf_cov)]:
    max_rc_dev = 0.0
    max_euler = 0.0
    max_identA = 0.0
    max_identB = 0.0
    max_rho_bar_dev = 0.0
    min_vol_margin_gmv = np.inf   # sigma_ERC - sigma_GMV, want >= 0
    min_vol_margin_ew = np.inf    # sigma_EW - sigma_ERC, want >= 0
    min_w = np.inf
    sweeps = []
    for asof in rdates:
        _, Sigma = GMV(cov_estimator=est_fn)._estimate(panel, asof)
        Sigma_np = Sigma.to_numpy()
        sigma_np = np.sqrt(np.diag(Sigma_np))
        n = Sigma_np.shape[0]

        w_erc, n_sweeps = _erc_long_only(Sigma, "mech5", asof)
        sweeps.append(n_sweeps)
        var = float(w_erc @ Sigma_np @ w_erc)
        sigma_p = np.sqrt(var)
        rc = (w_erc * (Sigma_np @ w_erc)) / sigma_p
        rc_share = rc / rc.sum()
        max_rc_dev = max(max_rc_dev, float(np.max(np.abs(rc_share - 1.0 / n))))
        max_euler = max(max_euler, abs(float(rc.sum()) - sigma_p))

        beta_p = (Sigma_np @ w_erc) / var
        max_identA = max(max_identA, float(np.max(np.abs(w_erc * beta_p - 1.0 / n))))

        D_inv = np.diag(1.0 / sigma_np)
        C = pd.DataFrame(D_inv @ Sigma_np @ D_inv, index=Sigma.columns, columns=Sigma.columns)
        w_erc_c, _ = _erc_long_only(C, "mech5", asof)
        x = (w_erc * sigma_np) / float(w_erc @ sigma_np)
        max_identB = max(max_identB, float(np.max(np.abs(x - w_erc_c))))

        C_np = C.to_numpy()
        rho_bar = (C_np.sum() - n) / (n * (n - 1))
        C_bar = np.full((n, n), rho_bar); np.fill_diagonal(C_bar, 1.0)
        Sigma_bar = np.diag(sigma_np) @ C_bar @ np.diag(sigma_np)
        Sigma_bar_df = pd.DataFrame(Sigma_bar, index=Sigma.columns, columns=Sigma.columns)
        w_erc_bar, _ = _erc_long_only(Sigma_bar_df, "mech5", asof)
        iv = (1.0 / sigma_np) / np.sum(1.0 / sigma_np)
        max_rho_bar_dev = max(max_rho_bar_dev, float(np.max(np.abs(w_erc_bar - iv))))

        w_gmv = _min_variance_long_only(Sigma)
        w_ew = np.full(n, 1.0 / n)

        def pv(w):
            return float(np.sqrt(w @ Sigma_np @ w))

        min_vol_margin_gmv = min(min_vol_margin_gmv, pv(w_erc) - pv(w_gmv))
        min_vol_margin_ew = min(min_vol_margin_ew, pv(w_ew) - pv(w_erc))
        min_w = min(min_w, float(w_erc.min()))

    mech5_rows.append({
        "estimator": est_name, "max_rc_share_dev": max_rc_dev, "max_euler": max_euler,
        "max_identA": max_identA, "max_identB": max_identB,
        "max_rho_bar_dev": max_rho_bar_dev,
        "min_vol_margin_gmv": min_vol_margin_gmv, "min_vol_margin_ew": min_vol_margin_ew,
        "min_w": min_w, "median_sweeps": float(np.median(sweeps)), "max_sweeps": int(np.max(sweeps)),
    })

mech5_table = pd.DataFrame(mech5_rows).set_index("estimator")
print(mech5_table.to_string(float_format=lambda v: f"{v:.3e}"))

for _, row in mech5_table.iterrows():
    assert row["max_rc_share_dev"] <= 1e-11, f"RC-share deviation failed: {row['max_rc_share_dev']}"
    assert row["max_euler"] <= 1e-12, f"Euler check failed: {row['max_euler']}"
    assert row["max_identA"] <= 1e-11, f"Identity A failed: {row['max_identA']}"
    assert row["max_identB"] <= 1e-11, f"Identity B failed: {row['max_identB']}"
    assert row["max_rho_bar_dev"] <= 1e-11, f"constant-correlation check failed: {row['max_rho_bar_dev']}"
    assert row["min_vol_margin_gmv"] >= -1e-11, f"vol ordering (GMV<=ERC) failed: {row['min_vol_margin_gmv']}"
    assert row["min_vol_margin_ew"] >= -1e-11, f"vol ordering (ERC<=EW) failed: {row['min_vol_margin_ew']}"
    assert row["min_w"] > 0, f"positivity failed: {row['min_w']}"

print("Mechanism checks: PASSED at all 208 rebalances, both estimators.")

n_sweeps_s = results["ERC(S)"]["strat"].sweeps
n_sweeps_lw = results["ERC(LW)"]["strat"].sweeps
print(f"strat.sweeps: ERC(S) median={np.median(n_sweeps_s):.0f} max={max(n_sweeps_s)}   "
      f"ERC(LW) median={np.median(n_sweeps_lw):.0f} max={max(n_sweeps_lw)}")

## §6 Hypothesis tests

### H1 — the UUP corner

In [ ]:
w_uup_erc_ts = results["ERC(S)"]["wlog"]["UUP"]
w_uup_gmv_ts = results["GMV(S)"]["wlog"]["UUP"]
w_uup_mdp_ts = results["MDP(S)"]["wlog"]["UUP"]

lt_gmv = w_uup_erc_ts < w_uup_gmv_ts
lt_mdp = w_uup_erc_ts < w_uup_mdp_ts
h1_full = lt_gmv & lt_mdp
n_full = len(h1_full)
test_mask = h1_full.index > split_ts
h1_test = h1_full.loc[test_mask]
n_test = int(test_mask.sum())

asof_2022 = pd.Timestamp("2022-12-31")
uup_2022 = {
    "ERC(S)": float(w_uup_erc_ts.loc[asof_2022]),
    "GMV(S)": float(w_uup_gmv_ts.loc[asof_2022]),
    "MDP(S)": float(w_uup_mdp_ts.loc[asof_2022]),
}

_, Sigma_2022c = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
Sigma_np_2022c = Sigma_2022c.to_numpy()
w_erc_2022_arr = results["ERC(S)"]["wlog"].loc[asof_2022].reindex(Sigma_2022c.columns).to_numpy()
var_2022c = float(w_erc_2022_arr @ Sigma_np_2022c @ w_erc_2022_arr)
beta_p_2022c = (Sigma_np_2022c @ w_erc_2022_arr) / var_2022c
iU2022c = list(Sigma_2022c.columns).index("UUP")

print(f"H1 (ERC<GMV and ERC<MDP): full {int(h1_full.sum())}/{n_full} "
      f"({100*h1_full.sum()/n_full:.1f}%), test {int(h1_test.sum())}/{n_test} "
      f"({100*h1_test.sum()/n_test:.1f}%)")
print(f"2022-12-31 w_UUP: ERC={uup_2022['ERC(S)']:.4f}  GMV={uup_2022['GMV(S)']:.4f}  MDP={uup_2022['MDP(S)']:.4f}")
print(f"2022-12-31 beta_UUP,p (ERC) = {beta_p_2022c[iU2022c]:.4f}")
print(f"holds at 2022-12-31: ERC<GMV={uup_2022['ERC(S)'] < uup_2022['GMV(S)']}  "
      f"ERC<MDP={uup_2022['ERC(S)'] < uup_2022['MDP(S)']}")

### H2 — effective N

In [ ]:
effN_erc_ts = 1.0 / (results["ERC(S)"]["wlog"] ** 2).sum(axis=1)
effN_gmv_ts = 1.0 / (results["GMV(S)"]["wlog"] ** 2).sum(axis=1)
effN_mdp_ts = 1.0 / (results["MDP(S)"]["wlog"] ** 2).sum(axis=1)

comparator_max = pd.concat([effN_gmv_ts, effN_mdp_ts], axis=1).max(axis=1)
h2_full = effN_erc_ts >= comparator_max
n_full2 = len(h2_full)
test_mask2 = h2_full.index > split_ts
h2_test = h2_full.loc[test_mask2]
n_test2 = int(test_mask2.sum())

print(f"H2 (effN(ERC) >= max(GMV,MDP)): full {int(h2_full.sum())}/{n_full2} "
      f"({100*h2_full.sum()/n_full2:.1f}%), test {int(h2_test.sum())}/{n_test2} "
      f"({100*h2_test.sum()/n_test2:.1f}%)")
print(f"median effN: ERC(S)={float(effN_erc_ts.median()):.4f}  "
      f"GMV(S)={float(effN_gmv_ts.median()):.4f}  MDP(S)={float(effN_mdp_ts.median()):.4f}")
print(f"2022-12-31 effN: ERC={float(effN_erc_ts.loc[asof_2022]):.4f}  "
      f"GMV={float(effN_gmv_ts.loc[asof_2022]):.4f}  MDP={float(effN_mdp_ts.loc[asof_2022]):.4f}")

### H3 / H4 / H5 — paired tests, &#946; decomposition, cost drag

Pairs: ERC(S) &#8722; GMV(S), ERC(S) &#8722; MDP(S), ERC(S) &#8722; EW,
ERC(LW) &#8722; ERC(S), ERC(S) &#8722; IV. Plain OLS (no HAC &#8212; same
caveat as notebooks 03&#8211;05), realized daily simple excess return on
the market's simple excess return; the identity check is 0 by OLS
construction (mean residual is exactly 0 for an OLS fit with an
intercept), not a residual to be interpreted.

In [ ]:
pairs = [
    ("ERC(S)", "GMV(S)"),
    ("ERC(S)", "MDP(S)"),
    ("ERC(S)", "EW"),
    ("ERC(LW)", "ERC(S)"),
    ("ERC(S)", "IV"),
]

paired_rows = []
paired_series = {}
for a_name, b_name in pairs:
    net_a = results[a_name]["net"]
    net_b = results[b_name]["net"]
    idx = net_a.index.intersection(net_b.index)
    diff = net_a.loc[idx] - net_b.loc[idx]
    paired_series[(a_name, b_name)] = diff
    win_defs = {
        "full": diff.index >= diff.index.min(),
        "train (<= split)": diff.index <= split_ts,
        "test (> split)": diff.index > split_ts,
    }
    for window_name, sel in win_defs.items():
        d = diff.loc[sel]
        ann_mean = float(d.mean() * ml.TRADING_DAYS)
        te = float(d.std(ddof=1) * np.sqrt(ml.TRADING_DAYS))
        t_stat = float(d.mean() / (d.std(ddof=1) / np.sqrt(len(d))))
        cost_drag_a = float(summary_table.loc[(a_name, window_name), "cost_drag"])
        cost_drag_b = float(summary_table.loc[(b_name, window_name), "cost_drag"])
        cost_drag_diff = cost_drag_a - cost_drag_b
        gross_diff = ann_mean + cost_drag_diff
        paired_rows.append({
            "pair": f"{a_name} - {b_name}", "window": window_name,
            "ann_mean_diff": ann_mean, "TE": te, "t": t_stat,
            "cost_drag_diff": cost_drag_diff, "gross_of_costs_mean_diff": gross_diff,
        })

paired_table = pd.DataFrame(paired_rows).set_index(["pair", "window"])
print(paired_table.round(4))

In [ ]:
print("cost drag %/yr per run (full window):")
for name in results:
    print(f"  {name:12s} {float(summary_table.loc[(name, 'full'), 'cost_drag']) * 100:.4f}%")

Single-strategy CAPM &#945;/&#946; below is **not pre-registered**
&#8212; shown for context only, no hypothesis is attached to any single
number in it, and it is Phase 2 multiple-testing material.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_rows = []
for name in ["ERC(S)", "GMV(S)", "MDP(S)", "EW", "ERC(LW)", "IV"]:
    net = results[name]["net"]
    idx = net.index.intersection(rf_daily.index).intersection(MKT_simple.index)
    strat_excess = net.loc[idx] - rf_daily.loc[idx]
    mkt_aligned = MKT_simple.loc[idx]
    fit = ols_with_intercept(strat_excess, mkt_aligned.to_frame("MKT"))
    capm_rows.append({
        "strategy": name,
        "alpha_ann": float(fit["coef"]["alpha"] * ml.TRADING_DAYS),
        "t_alpha": float(fit["tstat"]["alpha"]),
        "beta": float(fit["coef"]["MKT"]),
        "r2": fit["r2"], "n": fit["n"],
    })
capm_table = pd.DataFrame(capm_rows).set_index("strategy")
print(capm_table.round(4))

In [ ]:
# Direct regression of each pair's diff series on MKT: alpha_diff has its
# own t-stat (unlike alpha_a - alpha_b, which ignores the covariance
# between the two separate alpha estimates) -- this is what H3/H4/H5's
# "Delta-alpha |t| < 2" conditions are evaluated against.
diff_capm_rows = []
for a_name, b_name in pairs:
    diff = paired_series[(a_name, b_name)]
    idx = diff.index.intersection(MKT_simple.index)
    fit = ols_with_intercept(diff.loc[idx], MKT_simple.loc[idx].to_frame("MKT"))
    diff_capm_rows.append({
        "pair": f"{a_name} - {b_name}",
        "alpha_diff_ann": float(fit["coef"]["alpha"] * ml.TRADING_DAYS),
        "t_alpha_diff": float(fit["tstat"]["alpha"]),
        "beta_diff": float(fit["coef"]["MKT"]),
        "n": fit["n"],
    })
diff_capm_table = pd.DataFrame(diff_capm_rows).set_index("pair")
print(diff_capm_table.round(4))

In [ ]:
for a_name, b_name in pairs:
    alpha_a = float(capm_table.loc[a_name, "alpha_ann"]); beta_a = float(capm_table.loc[a_name, "beta"])
    alpha_b = float(capm_table.loc[b_name, "alpha_ann"]); beta_b = float(capm_table.loc[b_name, "beta"])

    net_a = results[a_name]["net"]
    idx = net_a.index.intersection(rf_daily.index).intersection(MKT_simple.index)
    mkt_ann_mean = float(MKT_simple.loc[idx].mean() * ml.TRADING_DAYS)

    d_alpha = alpha_a - alpha_b
    d_beta = beta_a - beta_b
    beta_part = d_beta * mkt_ann_mean
    decomposed = d_alpha + beta_part

    realized_diff_full = float(paired_table.loc[(f"{a_name} - {b_name}", "full"), "ann_mean_diff"])
    identity_check = realized_diff_full - decomposed

    pair_key = f"{a_name} - {b_name}"
    t_alpha_diff = float(diff_capm_table.loc[pair_key, "t_alpha_diff"])

    print(f"=== {pair_key} (full window) ===")
    print(f"  t_alpha[{a_name}]={float(capm_table.loc[a_name, 't_alpha']):.2f}  t_alpha[{b_name}]={float(capm_table.loc[b_name, 't_alpha']):.2f}")
    print(f"  d_alpha (ann)         = {d_alpha:.4%}   (direct diff-regression: {float(diff_capm_table.loc[pair_key, 'alpha_diff_ann']):.4%}, t={t_alpha_diff:.2f})")
    print(f"  d_beta                = {d_beta:.4f}")
    print(f"  ann mean MKT excess   = {mkt_ann_mean:.4%}")
    print(f"  beta_part = d_beta * mkt_ann_mean = {beta_part:.4%}")
    print(f"  d_alpha + beta_part   = {decomposed:.4%}")
    print(f"  realized ann_mean_diff (full) = {realized_diff_full:.4%}")
    print(f"  identity check (0 by OLS construction) = {identity_check:.4%}")
    print()

## §7 Figures

### F3 — group-weight stackplot

Which asset groups carry the target weight over time under ERC(S) vs.
MDP(S) vs. GMV(S)?

In [ ]:
group_order = list(ml.ASSET_GROUPS.keys())
group_colors = plt.cm.tab10(np.linspace(0, 1, len(group_order)))

def weights_by_group(wlog):
    grouped = wlog.T.groupby(ml.GROUP_OF).sum().T
    return grouped.reindex(columns=group_order).fillna(0.0)

erc_group = weights_by_group(results["ERC(S)"]["wlog"])
mdp_group = weights_by_group(results["MDP(S)"]["wlog"])
gmv_group = weights_by_group(results["GMV(S)"]["wlog"])

fig, axes = plt.subplots(3, 1, figsize=(9, 10), sharex=True)
for ax, grouped, title in zip(axes, [erc_group, mdp_group, gmv_group], ["ERC(S)", "MDP(S)", "GMV(S)"]):
    ax.stackplot(grouped.index, grouped.T.to_numpy(), labels=group_order, colors=group_colors)
    ax.axvline(split_ts, color="black", ls=":", lw=1)
    ax.set_ylim(0, 1)
    ax.set_title(f"{title} -- target weights by asset group")
axes[-1].set_xlabel("rebalance date")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
plt.tight_layout(); plt.show()

print("Mean target weight by asset group:")
group_means = pd.DataFrame({"ERC(S)": erc_group.mean(), "MDP(S)": mdp_group.mean(), "GMV(S)": gmv_group.mean()})
print(group_means.round(4))

### F4 — effN, max weight, and UUP weight over time

Is the effN/UUP contrast between ERC, MDP and GMV a snapshot artifact or
does it hold across the full 208-rebalance history?

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9.5, 11), sharex=True)

effN_ew_ts = 1.0 / (results["EW"]["wlog"] ** 2).sum(axis=1)
axes[0].plot(effN_erc_ts.index, effN_erc_ts.to_numpy(), label="ERC(S)", color="#d4a017", lw=1.2)
axes[0].plot(effN_mdp_ts.index, effN_mdp_ts.to_numpy(), label="MDP(S)", color=ml.FAMILY_COLORS["Risk-based"], lw=1.2)
axes[0].plot(effN_gmv_ts.index, effN_gmv_ts.to_numpy(), label="GMV(S)", color="#555555", lw=1.2)
axes[0].plot(effN_ew_ts.index, effN_ew_ts.to_numpy(), label="EW", color="#999999", lw=0.8, ls="--")
axes[0].axvline(split_ts, color="black", ls=":", lw=1)
axes[0].set_ylabel("effective N")
axes[0].legend(fontsize=8)

maxw_erc_ts = results["ERC(S)"]["wlog"].max(axis=1)
maxw_mdp_ts = results["MDP(S)"]["wlog"].max(axis=1)
maxw_gmv_ts = results["GMV(S)"]["wlog"].max(axis=1)
axes[1].plot(maxw_erc_ts.index, maxw_erc_ts.to_numpy() * 100, label="ERC(S)", color="#d4a017", lw=1.2)
axes[1].plot(maxw_mdp_ts.index, maxw_mdp_ts.to_numpy() * 100, label="MDP(S)", color=ml.FAMILY_COLORS["Risk-based"], lw=1.2)
axes[1].plot(maxw_gmv_ts.index, maxw_gmv_ts.to_numpy() * 100, label="GMV(S)", color="#555555", lw=1.2)
axes[1].axvline(split_ts, color="black", ls=":", lw=1)
axes[1].set_ylabel("max weight (%)")
axes[1].legend(fontsize=8)

axes[2].plot(w_uup_erc_ts.index, w_uup_erc_ts.to_numpy() * 100, label="ERC(S)", color="#d4a017", lw=1.2)
axes[2].plot(w_uup_mdp_ts.index, w_uup_mdp_ts.to_numpy() * 100, label="MDP(S)", color=ml.FAMILY_COLORS["Risk-based"], lw=1.2)
axes[2].plot(w_uup_gmv_ts.index, w_uup_gmv_ts.to_numpy() * 100, label="GMV(S)", color="#555555", lw=1.2)
axes[2].axvline(split_ts, color="black", ls=":", lw=1)
axes[2].set_ylabel("UUP weight (%)")
axes[2].set_xlabel("rebalance date")
axes[2].legend(fontsize=8)

fig.suptitle("ERC(S) vs MDP(S) vs GMV(S): effN, max weight, UUP weight")
plt.tight_layout(); plt.show()

### F5 — cumulative paired difference

For each registered pair, is the daily return difference a persistent
drift or a noisy path with no clear direction?

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5))
for a_name, b_name in pairs:
    diff = paired_series[(a_name, b_name)]
    cum = diff.cumsum() * 100
    ax.plot(cum.index, cum.to_numpy(), lw=1.1, label=f"{a_name} - {b_name}")
ax.axvline(split_ts, color="black", ls=":", lw=1, label="train/test split")
ax.axhline(0, color="black", lw=0.6)
ax.set_ylabel("cumulative daily return difference (%)")
ax.set_xlabel("date")
ax.set_title("Cumulative paired daily return differences")
ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## §8 TAKEAWAY

- **H1 — supported:** the UUP corner **shrinks** under ERC but does not
  disappear. w_UUP(ERC) &lt; w_UUP(GMV) and &lt; w_UUP(MDP) at 2022-12-31
  (48.36% vs 58.12% / 59.22%) and on 197/208 (94.7%) rebalances overall
  (test 33/40, 82.5%). At that same date &beta;<sub>UUP,p</sub>(ERC) =
  0.1591 &#8212; UUP is a weak hedge of the ERC portfolio itself, and by
  Identity A a low &beta;<sub>i,p</sub> is exactly what forces a large cash
  weight to still buy its mandated 1/13 share of risk. UUP's mean pairwise
  correlation with the rest of the universe at 2022-12-31 is &#8722;0.4659
  (it is also the single lowest-vol asset, rank 1/13, printed in &#167;3) &#8212;
  the same correlation-driven corner notebook 05 found for MDP. In
  **absolute** terms 48.36% in one asset is still a corner: ERC does not
  spread the portfolio evenly across the universe, it spreads *risk*
  evenly, and UUP's low correlation to everything else means a small
  amount of its risk buys a large amount of its notional.
- **The equal-risk-share result is an identity, not a finding.** Every
  ERC portfolio in this notebook has RC share = 1/13 for all 13 assets by
  *construction* &#8212; &#167;5 confirms this holds to &#8804; 1e-11 across
  all 208 rebalances &times; both estimators because the solver enforces
  it, not because of anything discovered about this universe. The actual
  empirical content of this notebook is what ERC does to *cash* weights and
  concentration, not that risk shares come out equal.
- **H2 — supported, decisively:** effN(ERC) &#8805; max(effN GMV, effN MDP)
  on 208/208 (100%) rebalances, test 40/40 (100%); median effN
  ERC(S) = 6.86 vs GMV(S) = 3.79, MDP(S) = 3.35. But even with risk shares
  *exactly* equal, effN(ERC) at 2022-12-31 is 3.88 &#8212; above GMV's 2.51
  and MDP's 2.47, but nowhere near EW's 13.00. Equalizing risk
  contribution is not the same thing as equalizing capital, and it is not a
  general cure for concentration when one asset (UUP) is both very
  low-vol and strongly anti-correlated with the rest of the book.
- **H3 — null holds (on the registered statistic):** &Delta;&alpha; from the
  direct diff-regression &#8212; the pre-registered H3 statistic &#8212; is
  +0.09%/yr for ERC(S) &#8722; GMV(S) (t = 0.29) and &#8722;0.26%/yr for
  ERC(S) &#8722; MDP(S) (t = &#8722;0.74); both &#124;t&#124; &lt; 2, so the
  null holds for both pairs. Descriptively, and explicitly **not** as a
  result (per the pre-registration, the total paired difference carries no
  directional claim): the *total* paired difference for ERC(S) &#8722;
  GMV(S) is +1.04%/yr full (t = 2.47, exceeding the &#124;t&#124; &lt; 2
  band) but +1.14%/yr test (t = 1.32, inside it), while ERC(S) &#8722;
  MDP(S)'s total is +0.40%/yr full (t = 0.99) and +0.04%/yr test
  (t = 0.06). The &#946; decomposition shows the ERC&#8722;GMV total gap is
  almost entirely the beta part (&Delta;&beta; part = +0.95%/yr vs.
  &Delta;&alpha; = +0.09%/yr) &#8212; consistent with ERC's higher ex-ante
  vol on the same &Sigma;, not with anything about the ERC construction
  itself, which is exactly why H3 was registered on &Delta;&alpha; rather
  than on the total.
- **H4 — supported, borderline (same pattern as notebook 05):** ERC(S)
  &#8722; EW is &#8722;2.78%/yr full (t = &#8722;2.08, just past the
  &#8722;2 threshold) and the test-window mean is also negative
  (&#8722;3.16%/yr, though its own t is only &#8722;1.15).
  &#124;&Delta;&beta; part&#124; = 3.53%/yr &gt; &#124;&Delta;&alpha;&#124;
  = 0.76%/yr, and &Delta;&alpha;'s own t (direct diff-regression) = 0.89
  &lt; 2 &#8212; consistent with a beta-driven shortfall, as the hypothesis
  predicted, not an alpha effect. The full-window t sits only marginally
  past &#8722;2, so this reads as a marginal pass, not a decisive one.
- **H5 — both nulls hold, one narrowly:** ERC(LW) &#8722; ERC(S) is
  +0.18%/yr full (t = 1.75) and +0.24%/yr test (t = 1.24) &#8212; both
  &#124;t&#124; &lt; 2, but the full-window t sits close to the 2 threshold,
  so this null holds narrowly rather than cleanly (the same word notebook
  05 used for its own t = 1.98 null). ERC(S) &#8722; IV &Delta;&alpha;
  (direct diff-regression) is +0.33%/yr (t = 0.56), comfortably inside the
  null band. It is also worth disclosing why ERC(LW) was backtested as a
  sensitivity at all: the read-only design check found
  max half-L1(ERC(LW), ERC(S)) = 0.0307 over the three snapshot dates
  against a pre-set 3% threshold &#8212; it cleared that bar by about 7 bp,
  on one of the three dates. That is a weak basis for treating the LW
  sensitivity as a meaningfully different portfolio in the first place, so
  its narrow null here should be read as consistent with "there was barely
  a difference to test," not as a strong robustness result.
- **Cost drag is small everywhere** (0.01&#8211;0.07%/yr across all six
  runs) and does not change any of the above conclusions; ERC(S)'s own
  turnover (26.6%/yr) sits between EW's (13.6%/yr) and MDP(S)'s (52.5%/yr).
- **nb05 link:** notebook 05 found MDP keeps the UUP corner that survives
  covariance shrinkage; this notebook finds ERC shrinks that corner in
  relative terms (48% vs. 58&#8211;59%) but does not remove it, and the
  mechanism is legible for the first time via Identity A &#8212; a
  low-&beta;-to-portfolio asset needs a large cash weight to carry an equal
  share of risk. No claim beyond what is tested here is made: this
  notebook does not test whether the corner is "wrong," does not compare
  ERC against a risk-budgeted (non-equal) alternative, and does not
  attribute UUP's low &beta;<sub>i,p</sub> to any specific macro regime
  &#8212; those are Phase 2 questions.